# 39 · Retries without double side effects; resumable runs
Level L4 · Part 5 · professional · ~35 min · builds on 28, 35

## Goal
Give Scout's side-effecting tools an **idempotency key** so a retried call never writes twice, save a
**run checkpoint** after every step, then kill Scout right after `write_file` and resume the run:
no completed tool runs again and `mars_day.txt` is written exactly once.

## The idea
Some tools only read (`search_notes`, `calculator`): running them twice is harmless. Others change the
world: they save a file, send an email, charge a card. If the reply of such a tool is lost and you
simply run it again, it happens twice. The fix is an **idempotency key**: a name for *this one call*,
sent with every attempt. The tool remembers the keys it has seen and answers a repeat with the saved
result instead of acting again. Retry the call, not the tool.

The second half is a **run checkpoint**: the message list saved to disk after every step. A killed run
reloads it and carries on from the last step instead of starting over.

Analogy: a numbered cheque. If the bank sees cheque 0042 twice, it pays once. And a bookmark: after a
power cut you open the book at the bookmark, not at page 1.

## Picture
```mermaid
sequenceDiagram
    participant L as run_agent_resumable
    participant C as checkpoint file
    participant T as write_file + key ledger
    L->>C: save after model turn (call write_file)
    L->>T: write_file, key run-42:step3:call0
    T-->>T: write file, remember key
    Note over L: killed before the result is saved
    L->>C: resume: load messages
    L->>T: same call, same key
    T-->>L: saved result, no second write
```

## Step by step
### 1. Setup from lesson 35 (shortened)
The six notes, `search_notes` and the safe `calculator`. This lesson needs no real model: Scout's
replies come from the scripted fake, so there is no recorder.

In [1]:
import ast, json, operator, os, shutil, uuid
from collections import Counter
from pathlib import Path

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
def search_notes(query: str) -> list[dict]:
    q = query.lower()
    return [n for n in NOTES if q in n["text"].lower()]

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}
def evaluate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
    raise ValueError("use numbers, + - * / and brackets, e.g. '24*60+39'")
def calculator(expression: str) -> float:
    return evaluate(ast.parse(expression, mode="eval").body)

print(search_notes("day on Mars"), calculator("24*60+39"))

[{'id': 3, 'text': 'A day on Mars lasts about 24 hours and 39 minutes.'}] 1479


Scout's folder and a folder for run checkpoints, both emptied by `reset()`. `EFFECTS` counts how many
times a side effect *really* happened, so we can prove "exactly once" later. `write_file` is lesson
35's, shortened, plus one counter line.

In [2]:
SCOUT_DIR, RUNS_DIR = Path("../data/scout_files_39"), Path("../data/runs_39")
EFFECTS = Counter()                                          # real side effects, by tool name

def reset():
    for folder in (SCOUT_DIR, RUNS_DIR):
        shutil.rmtree(folder, ignore_errors=True)
        folder.mkdir(parents=True)
    EFFECTS.clear()

def write_file(filename: str, text: str, overwrite: bool = False) -> str:
    target = (SCOUT_DIR / filename).resolve()
    if target.parent != SCOUT_DIR.resolve():
        raise ValueError(f"'{filename}' is outside Scout's folder. Use a plain file name.")
    if target.exists() and not overwrite:
        raise FileExistsError(f"'{filename}' already exists. Pick a new name or set overwrite to true.")
    target.write_text(text)
    EFFECTS["write_file"] += 1
    return f"saved {len(text)} characters to {target.name}"

reset()
print(write_file("test.txt", "hello"), "|", dict(EFFECTS))

saved 5 characters to test.txt | {'write_file': 1}


Lesson 35's `ScriptedModel` (with its history check) and `run_tool`, unchanged apart from dropping
the result cap.

In [3]:
def check_history(messages: list[dict]) -> None:
    issued = set()
    for m in messages:
        issued |= {c["id"] for c in m.get("tool_calls", [])}
        if m["role"] == "tool" and m["call_id"] not in issued:
            raise ValueError(f"tool result for unknown call id {m['call_id']!r}")

class ScriptedModel:
    def __init__(self, script: list):
        self.script, self.calls, self.next_id = script, 0, 1
    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        check_history(messages)
        self.calls += 1
        if self.calls > 50:
            raise RuntimeError("model called 50 times: the loop never stopped")
        reply = self.script[min(self.calls, len(self.script)) - 1]
        reply = reply(messages) if callable(reply) else json.loads(json.dumps(reply))
        for call in reply.get("tool_calls", []):
            call["id"], self.next_id = f"call_{self.next_id}", self.next_id + 1
        return {"role": "assistant", **reply}

`run_tool` never raises (lesson 28): any exception becomes an error result.

In [4]:
def run_tool(call: dict, tools: dict) -> dict:
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available tools: {', '.join(tools)}.",
                "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

print(run_tool({"name": "calculator", "args": {"expression": "1/0"}}, {"calculator": calculator}))

{'content': 'ZeroDivisionError: division by zero', 'is_error': True}


### 2. Which tools are safe to repeat?
A new tool: `append_file` adds one line to Scout's research journal. Then we run every tool twice
with the same arguments and look at the second answer.

In [5]:
def append_file(filename: str, text: str) -> str:
    target = (SCOUT_DIR / filename).resolve()
    if target.parent != SCOUT_DIR.resolve():
        raise ValueError(f"'{filename}' is outside Scout's folder. Use a plain file name.")
    with target.open("a") as f:
        f.write(text + "\n")
    EFFECTS["append_file"] += 1
    return f"added 1 line to {target.name}"

def lines(filename: str) -> list[str]:
    path = SCOUT_DIR / filename
    return path.read_text().splitlines() if path.exists() else []

TOOLS = {"search_notes": search_notes, "calculator": calculator,
         "write_file": write_file, "append_file": append_file}
SCHEMAS = [{"name": name} for name in TOOLS]                 # the fake model never reads schemas
print("tools:", list(TOOLS))

tools: ['search_notes', 'calculator', 'write_file', 'append_file']


Each call twice. The read-only tools give the same answer. `write_file` with `overwrite` true leaves
the same file: it is **idempotent** (doing it twice equals doing it once).

In [6]:
TWICE = [("search_notes", {"query": "day on Mars"}), ("calculator", {"expression": "24*60+39"}),
         ("write_file", {"filename": "a.txt", "text": "hi", "overwrite": True}),
         ("write_file", {"filename": "b.txt", "text": "hi"}),
         ("append_file", {"filename": "journal.txt", "text": "Mars day: 1479 min"})]
for name, args in TWICE:
    first, second = (run_tool({"name": name, "args": args}, TOOLS) for _ in range(2))
    print(f"{name:12} 2nd call -> {second['content'][:62]}")
print("journal.txt:", lines("journal.txt"))
reset()

search_notes 2nd call -> [{"id": 3, "text": "A day on Mars lasts about 24 hours and 39 
calculator   2nd call -> 1479
write_file   2nd call -> "saved 2 characters to a.txt"
write_file   2nd call -> FileExistsError: 'b.txt' already exists. Pick a new name or se
append_file  2nd call -> "added 1 line to journal.txt"
journal.txt: ['Mars day: 1479 min', 'Mars day: 1479 min']


Two different failures. `write_file` without `overwrite` reports an error on the repeat, so the model
is told the save *failed* although the first call worked. `append_file` silently wrote the line twice.
Both are **duplicate side effects**: the effect of one intended call, applied twice.

### 3. The lost reply
Retries are where repeats come from. A timeout tells you the reply did not arrive, not that the work
did not happen. `lose_first_reply` simulates that: the tool does its work, then the first reply is
lost. `call_with_retry` takes a function that makes one attempt and repeats it after a timeout.

In [7]:
def lose_first_reply(fn):
    lost = []
    def wrapper(**args):
        result = fn(**args)                                  # the work is done...
        if not lost:
            lost.append(True)
            raise TimeoutError("no reply within 10 s")       # ...but the reply never arrives
        return result
    wrapper.__name__ = fn.__name__
    return wrapper

def call_with_retry(attempt, tries: int = 3) -> dict:
    for n in range(1, tries + 1):
        result = attempt()
        if not result["content"].startswith("TimeoutError"):
            return result
        print(f"attempt {n}: {result['content']} -> retry")
    return result

The naive retry repeats the tool. The journal gets two lines.

In [8]:
JOURNAL = {"name": "append_file", "args": {"filename": "journal.txt", "text": "Mars day: 1479 min"}}
flaky = {**TOOLS, "append_file": lose_first_reply(append_file)}
print(call_with_retry(lambda: run_tool(JOURNAL, flaky)))
print("journal.txt:", lines("journal.txt"), "| real appends:", EFFECTS["append_file"])
reset()

attempt 1: TimeoutError: no reply within 10 s -> retry
{'content': '"added 1 line to journal.txt"', 'is_error': False}
journal.txt: ['Mars day: 1479 min', 'Mars day: 1479 min'] | real appends: 2


### 4. An idempotency key, checked by the tool
`idempotent` wraps a tool. With a key it looks the key up in a small ledger file in Scout's folder.
A new key: do the work, then save the key, its arguments and the result. A known key: return the
saved result and do nothing. A known key with *different* arguments is a bug in the caller, so it
raises (Stripe's API does the same). The check lives on the tool's side, because only the tool knows
whether the work already happened.

In [9]:
LEDGER = SCOUT_DIR / ".keys.json"

def idempotent(fn):
    def wrapper(idempotency_key: str | None = None, **args):
        if idempotency_key is None:
            return fn(**args)
        seen = json.loads(LEDGER.read_text()) if LEDGER.exists() else {}
        if idempotency_key in seen:
            if seen[idempotency_key]["args"] != args:
                raise ValueError(f"key {idempotency_key!r} was already used with other arguments")
            print(f"  {fn.__name__}: key {idempotency_key!r} seen before -> saved result, no new effect")
            return seen[idempotency_key]["result"]
        result = fn(**args)
        seen[idempotency_key] = {"args": args, "result": result}
        LEDGER.write_text(json.dumps(seen))
        return result
    wrapper.__name__ = fn.__name__
    return wrapper

The loop adds the key; the model never sees it. `run_tool_keyed` puts the key into the arguments of
side-effecting tools only. Read-only tools need no key.

In [10]:
SIDE_EFFECTS = {"write_file", "append_file"}
TOOLS_ONCE = {**TOOLS, "write_file": idempotent(write_file), "append_file": idempotent(append_file)}

def run_tool_keyed(call: dict, tools: dict, key: str) -> dict:
    if call["name"] in SIDE_EFFECTS:
        call = {**call, "args": {**call["args"], "idempotency_key": key}}
    return run_tool(call, tools)

The same lost reply, now with a key. Every attempt sends the *same* key, because the key belongs to
the call. Then one call reuses the key with other text.

In [11]:
flaky = {**TOOLS_ONCE, "append_file": lose_first_reply(TOOLS_ONCE["append_file"])}
print(call_with_retry(lambda: run_tool_keyed(JOURNAL, flaky, key="demo-1")))
print("journal.txt:", lines("journal.txt"), "| real appends:", EFFECTS["append_file"])
other = {"name": "append_file", "args": {"filename": "journal.txt", "text": "something else"}}
print(run_tool_keyed(other, TOOLS_ONCE, key="demo-1"))
reset()

attempt 1: TimeoutError: no reply within 10 s -> retry
  append_file: key 'demo-1' seen before -> saved result, no new effect
{'content': '"added 1 line to journal.txt"', 'is_error': False}
journal.txt: ['Mars day: 1479 min'] | real appends: 1
{'content': "ValueError: key 'demo-1' was already used with other arguments", 'is_error': True}


One line, one real append, and a clear error for the misused key. One gap remains: a crash *between*
the work and `LEDGER.write_text` would still repeat the work. Real services close it by saving the
key in the same database transaction as the effect.

### 5. Checkpoint the run after every step
A run checkpoint is the run id plus the message list, as JSON. `save_checkpoint` writes a temporary
file and then renames it over the old one with `os.replace`. A rename is atomic: a crash leaves the
old checkpoint or the new one, never half a file.

In [12]:
def save_checkpoint(run_id: str, state: dict) -> None:
    path = RUNS_DIR / f"{run_id}.json"
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(state))
    os.replace(tmp, path)                                    # atomic swap

def load_checkpoint(run_id: str) -> dict | None:
    path = RUNS_DIR / f"{run_id}.json"
    return json.loads(path.read_text()) if path.exists() else None

save_checkpoint("demo", {"run_id": "demo", "messages": [{"role": "user", "text": "hi"}]})
print(load_checkpoint("demo"), "|", load_checkpoint("nope"))

{'run_id': 'demo', 'messages': [{'role': 'user', 'text': 'hi'}]} | None


### 6. Find the calls that still need a result
After a crash, the last model turn may ask for tools whose results were never saved. `pending_calls`
returns them with their position in the turn.

In [13]:
def pending_calls(messages: list[dict]) -> list[tuple[int, dict]]:
    turns = [i for i, m in enumerate(messages) if m["role"] == "assistant"]
    if not turns:
        return []
    last = turns[-1]
    answered = {m["call_id"] for m in messages[last + 1:] if m["role"] == "tool"}
    return [(i, c) for i, c in enumerate(messages[last].get("tool_calls", [])) if c["id"] not in answered]

turn = {"role": "assistant", "tool_calls": [{"id": "call_1", "name": "a", "args": {}},
                                            {"id": "call_2", "name": "b", "args": {}}]}
done = {"role": "tool", "call_id": "call_1", "name": "a", "content": "1", "is_error": False}
print(pending_calls([{"role": "user", "text": "q"}, turn, done]))

[(1, {'id': 'call_2', 'name': 'b', 'args': {}})]


### 7. A loop that can resume
`run_agent_resumable` starts from the checkpoint if there is one. Each pass runs any pending calls,
then asks the model. It saves after the model turn and after every tool result. The key is
`run_id:step:call`: the step number and position come from the saved history, so a resumed run
rebuilds exactly the same key (Temporal suggests the same shape: run id plus activity id).

In [14]:
def run_agent_resumable(run_id, question, model, tools, schemas, max_steps=8) -> dict:
    state = load_checkpoint(run_id) or {"run_id": run_id, "messages": [{"role": "user", "text": question}]}
    messages = state["messages"]
    print(f"{run_id}: {'resuming' if len(messages) > 1 else 'new run'} with {len(messages)} messages")
    while True:
        step = sum(m["role"] == "assistant" for m in messages)
        for i, call in pending_calls(messages):
            result = run_tool_keyed(call, tools, key=f"{run_id}:step{step}:call{i}")
            print(f"step {step}: {call['name']}({call['args']}) -> {result['content'][:50]}")
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
            save_checkpoint(run_id, state)
        if messages[-1]["role"] == "assistant":              # a text answer: the run is over
            return {"answer": messages[-1]["text"], "stop": "end_of_turn", "steps": step, "messages": messages}
        if step == max_steps:
            return {"answer": None, "stop": "step_limit", "steps": step, "messages": messages}
        messages.append(model(messages, schemas))
        save_checkpoint(run_id, state)

### 8. Scout's task, decided from the history
A real model is stateless (lesson 08): it decides from the messages alone. Our fake does the same, so
a fresh copy after a restart picks up where the old one stopped. `fresh_model` also starts the fake's
call ids after those already in the history; a real provider's ids are unique anyway.

In [15]:
QUESTION = "How many minutes long is a day on Mars? Save it to mars_day.txt and note it in journal.txt."
def tool(name, **args): return {"tool_calls": [{"name": name, "args": args}]}

def scout_policy(messages):
    results = [json.loads(m["content"]) for m in messages if m["role"] == "tool"]
    if len(results) == 0: return tool("search_notes", query="day on Mars")
    if len(results) == 1: return tool("calculator", expression="24*60+39")
    if len(results) == 2: return tool("write_file", filename="mars_day.txt",
                                      text=f"A day on Mars lasts about {results[1]} minutes.")
    if len(results) == 3: return tool("append_file", filename="journal.txt", text="saved mars_day.txt")
    return {"text": f"A day on Mars lasts about {results[1]} minutes; saved and noted in the journal."}

def fresh_model(messages=()):
    model = ScriptedModel([scout_policy])
    model.next_id = 1 + sum(len(m.get("tool_calls", [])) for m in messages)
    return model

A clean run first, to see the trace and the checkpoint file.

In [16]:
run = run_agent_resumable("run-ok", QUESTION, fresh_model(), TOOLS_ONCE, SCHEMAS)
print(run["stop"], "|", run["answer"])
print("checkpoint:", len(load_checkpoint("run-ok")["messages"]), "messages | real effects:", dict(EFFECTS))
reset()

run-ok: new run with 1 messages
step 1: search_notes({'query': 'day on Mars'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 h
step 2: calculator({'expression': '24*60+39'}) -> 1479
step 3: write_file({'filename': 'mars_day.txt', 'text': 'A day on Mars lasts about 1479 minutes.'}) -> "saved 39 characters to mars_day.txt"
step 4: append_file({'filename': 'journal.txt', 'text': 'saved mars_day.txt'}) -> "added 1 line to journal.txt"
end_of_turn | A day on Mars lasts about 1479 minutes; saved and noted in the journal.
checkpoint: 10 messages | real effects: {'write_file': 1, 'append_file': 1}


### 9. Kill Scout right after `write_file`
`Killed` stands in for `kill -9` or a crashed machine. It subclasses `BaseException`, so `run_tool`'s
`except Exception` cannot catch it, just as no `except` survives a dead process. `die_after` lets the
tool finish its work and then kills the run before the result can be checkpointed.

In [17]:
class Killed(BaseException):
    pass

def die_after(fn):
    def wrapper(**args):
        fn(**args)
        raise Killed(f"process killed right after {fn.__name__} finished")
    return wrapper

doomed = {**TOOLS_ONCE, "write_file": die_after(TOOLS_ONCE["write_file"])}
try:
    run_agent_resumable("run-42", QUESTION, fresh_model(), doomed, SCHEMAS)
except Killed as e:
    print("KILLED:", e)

run-42: new run with 1 messages
step 1: search_notes({'query': 'day on Mars'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 h
step 2: calculator({'expression': '24*60+39'}) -> 1479
KILLED: process killed right after write_file finished


What survived on disk: the file, the ledger entry, and a checkpoint whose last message is the model's
`write_file` request with no result.

In [18]:
saved = load_checkpoint("run-42")
print("files:", sorted(p.name for p in SCOUT_DIR.iterdir()), "| real effects:", dict(EFFECTS))
print("last checkpointed message:", saved["messages"][-1]["tool_calls"][0]["name"])
print("pending after restart:", [c["name"] for _, c in pending_calls(saved["messages"])])
print("keys in ledger:", list(json.loads(LEDGER.read_text())))

files: ['.keys.json', 'mars_day.txt'] | real effects: {'write_file': 1}
last checkpointed message: write_file
pending after restart: ['write_file']
keys in ledger: ['run-42:step3:call0']


Without the key, resuming would run that pending call on plain `write_file`. The file already exists,
so the model would be told its save failed. (This only reads the result; nothing is saved.)

In [19]:
pending = pending_calls(saved["messages"])[0][1]
print(run_tool(pending, TOOLS))

{'content': "FileExistsError: 'mars_day.txt' already exists. Pick a new name or set overwrite to true.", 'is_error': True}


### 10. Resume
A new process: a fresh fake model, the normal tools, the same run id. The loop loads the checkpoint,
finds the pending `write_file` call, rebuilds its key and sends it again.

In [20]:
run = run_agent_resumable("run-42", QUESTION, fresh_model(saved["messages"]), TOOLS_ONCE, SCHEMAS)
print(run["stop"], "| steps:", run["steps"], "|", run["answer"])
print("real effects:", dict(EFFECTS), "| journal.txt:", lines("journal.txt"))
print("mars_day.txt:", (SCOUT_DIR / "mars_day.txt").read_text())

run-42: resuming with 6 messages
  write_file: key 'run-42:step3:call0' seen before -> saved result, no new effect
step 3: write_file({'filename': 'mars_day.txt', 'text': 'A day on Mars lasts about 1479 minutes.'}) -> "saved 39 characters to mars_day.txt"
step 4: append_file({'filename': 'journal.txt', 'text': 'saved mars_day.txt'}) -> "added 1 line to journal.txt"
end_of_turn | steps: 5 | A day on Mars lasts about 1479 minutes; saved and noted in the journal.
real effects: {'write_file': 1, 'append_file': 1} | journal.txt: ['saved mars_day.txt']
mars_day.txt: A day on Mars lasts about 1479 minutes.


Read the trace. The resumed run did not search or calculate again: those results were in the
checkpoint. The `write_file` call reused key `run-42:step3:call0`, the ledger answered, and the file
was written once in total. Then the run went on to the journal and the answer.

## At work
- **Payment and messaging APIs take an idempotency key per request.** Stripe returns the saved result
  of the first request for every repeat of its key (even a 500), rejects a key reused with other
  parameters, and may prune keys after 24 hours. Risky calls also wait for a person. → lesson 52
- **Durable-execution engines retry steps, so steps must be idempotent.** Temporal warns an Activity
  "may be executed multiple times" and suggests run id + activity id as the key. Resuming runs across
  a provider outage → lesson 57
- **Checkpoints need real storage and cleanup.** LangGraph's docs warn that in-memory checkpointers
  lose everything on restart and that checkpoints pile up and need pruning. → lesson 55
- **Log the run id and key with every tool call**, so a duplicate can be traced to one retry. → lesson 49

## What just happened
- Read-only tools are safe to repeat; `append_file` and `write_file` without `overwrite` are not.
- A lost reply plus a naive retry wrote the journal line twice: a duplicate side effect.
- With the same idempotency key on every attempt, the tool's ledger returned the saved result and the
  line was written once. A key reused with other arguments raised.
- The resumable loop saved a checkpoint after each model turn and each tool result, using an atomic
  rename.
- Killed after `write_file`, the run resumed from the checkpoint, rebuilt the same key, skipped the
  second write, and finished with every side effect counted exactly once.

## Common mistakes
**1. A new key for every attempt.** Making the key inside the retried function (here `uuid4()` in the
lambda) means each attempt looks like a new call to the tool. The key protects nothing.

In [21]:
reset()
flaky = {**TOOLS_ONCE, "append_file": lose_first_reply(TOOLS_ONCE["append_file"])}
call_with_retry(lambda: run_tool_keyed(JOURNAL, flaky, key=str(uuid.uuid4())))
assert len(lines("journal.txt")) == 1, f"journal has {len(lines('journal.txt'))} lines: {lines('journal.txt')}"

attempt 1: TimeoutError: no reply within 10 s -> retry


AssertionError: journal has 2 lines: ['Mars day: 1479 min', 'Mars day: 1479 min']

Fix: make the key once per call, from things that stay the same across retries and restarts (run id,
step, position), and pass that same key to every attempt.

In [22]:
reset()
flaky = {**TOOLS_ONCE, "append_file": lose_first_reply(TOOLS_ONCE["append_file"])}
key = "run-7:step4:call0"                                    # fixed before the first attempt
call_with_retry(lambda: run_tool_keyed(JOURNAL, flaky, key=key))
print("journal.txt:", lines("journal.txt"))

attempt 1: TimeoutError: no reply within 10 s -> retry
  append_file: key 'run-7:step4:call0' seen before -> saved result, no new effect
journal.txt: ['Mars day: 1479 min']


**2. Writing the checkpoint in place.** `path.write_text` first empties the file, then writes. A crash
in between leaves half a JSON document, and the run can no longer be resumed at all.

In [23]:
path = RUNS_DIR / "run-7.json"
text = json.dumps(saved)
path.write_text(text[: len(text) // 2])                      # the process died halfway through
load_checkpoint("run-7")

JSONDecodeError: Unterminated string starting at: line 1 column 344 (char 343)

Fix: write a temporary file and `os.replace` it, as `save_checkpoint` does. A crash while writing the
temporary file leaves the last good checkpoint untouched.

In [24]:
save_checkpoint("run-7", saved)                              # a good checkpoint
(RUNS_DIR / "run-7.tmp").write_text(text[: len(text) // 2])  # crash while writing the next one
print("still loads:", len(load_checkpoint("run-7")["messages"]), "messages")

still loads: 6 messages


## Try it
Kill Scout right after `append_file` instead (its fourth tool), with run id `"run-99"`. Then resume.
Check that the run ends with `end_of_turn`, `journal.txt` has exactly one line, and `EFFECTS` shows
one `write_file` and one `append_file`. Bonus: print the key the resumed `append_file` call used.

In [25]:
reset()
# your code here

## Key terms
- **idempotent** — doing an operation twice has the same effect as doing it once; reading a note is, appending a line is not.
- **duplicate side effect** — the effect of one intended tool call applied twice, usually because a retry or a restart repeated it.
- **idempotency key** — a name for one call, sent with every attempt; the tool remembers it and answers a repeat with the saved result instead of acting again.
- **run checkpoint** — the run id and message list saved to disk after every step, so a killed run can continue instead of starting over.
- **resume** — reload a run's checkpoint, run the tool calls that still have no result (with their original keys), and continue the loop.
- **atomic write** — write to a temporary file, then rename it over the old one, so a crash leaves either the old or the new file, never half of one.